# A. Data Simulation

Having established the theoretical framework for continuous-variable quantum systems — ranging from the quantization of the electromagnetic field to the formalism of Gaussian states and entanglement criteria — we now turn to the numerical application of these concepts. While analytical solutions offer deep insight into the fundamental properties of quantum states, computational simulations are an indispensable tool for analyzing complex optical networks and modeling experimental outcomes. In this notebook we translate the mathematical descriptions of quantum states into executable code.

**Simulating measurement statistics.** In an actual laboratory setting, quantum states are not accessed directly; they are reconstructed via statistical analysis of measurement data. By generating random numbers from specific probability distributions, we can simulate these results.

We recommend typing the code yourself as you read, e.g. in Google Colab, so you don't need a complicated setup.

> **Convention.** Throughout this notebook the quadrature operator is
> $$\hat X_\varphi = \hat a\, e^{i\varphi} + \hat a^\dagger e^{-i\varphi},$$
> so the vacuum has $\mathrm{Var}[\hat X_\varphi] = 1$. In the units $\hbar = 2$ used later with Strawberry Fields, $\hat X_0 = \hat x$ and $\hat X_{\pi/2} = -\hat p$, which is why $[\hat X_\varphi, \hat X_{\varphi+\pi/2}] = -2i$ below.

## Generating quadrature data

To generate random numbers representing a state, we need its cumulative distribution function (CDF) $F(x,\varphi)$. For the specific state to be generated, the characteristic function $\chi(\beta)$ must be known. The CDF can be calculated from it in two steps.

1. Calculate the quadrature probability density function $p(x,\varphi)$ for the desired state:
$$p(x,\varphi) = \frac{1}{2\pi}\int_{-\infty}^{\infty} dy\; e^{-iyx}\,\chi\!\left(iy\,e^{-i\varphi}\right).$$

2. Integrate the density $p(x,\varphi)$ up to $x$:
$$F(x,\varphi) = \int_{-\infty}^{x} d\tilde x\; p(\tilde x,\varphi).$$

### Example: thermal state

A thermal state is parametrized by the mean number of thermal photons, $\bar n = \left[\exp\!\left(\frac{\hbar\omega}{k_B T}\right) - 1\right]^{-1}$. We get:

$$\chi_{th}(\beta) = \exp\!\left[-\left(\bar n + \tfrac{1}{2}\right)|\beta|^2\right]$$

$$p(x,\varphi) = \frac{1}{2\pi}\int_{-\infty}^{\infty} dy\; e^{-iyx}\, e^{-\bar n y^2 - y^2/2} = \frac{1}{\sqrt{\pi(4\bar n+2)}}\exp\!\left(-\frac{x^2}{4\bar n+2}\right)$$

$$F(x,\varphi) = \int_{-\infty}^{x} d\tilde x\;\frac{1}{\sqrt{\pi(4\bar n+2)}}\exp\!\left(-\frac{\tilde x^2}{4\bar n+2}\right) = \frac{1}{2} + \frac{1}{2}\,\mathrm{erf}\!\left(\frac{x}{\sqrt{4\bar n+2}}\right)$$

### Sampling procedure

Now we generate two sets (of the same size) of uniformly distributed random numbers. One set, over the interval $[-\pi,\pi)$, represents the phases $\varphi_j$; the other, over $[0,1]$, gives the values $F(x_j,\varphi_j)$. By numerically solving equations of the type $F(x_j,\varphi_j) = p_0 \in [0,1]$ for a fixed $\varphi = \varphi_j$ with respect to $x_j$, we obtain the corresponding quadrature values $x_j$.

*Note:* since $p(x,\varphi+\pi) = p(-x,\varphi)$, the interval $[0,\pi)$ would already contain all the information. We use $[-\pi,\pi)$ so that the plots show two full periods.

---
### Task 1
Perform the simulation of the data for **one** of the states listed below.

- **Fock states** $|n\rangle$ (for one $n$ from the interval $2 \le n \le 10$):
$$\chi_n(\beta) = \exp\!\left[-\frac{|\beta|^2}{2}\right] L_n\!\left(|\beta|^2\right),$$
where $L_n(z)$ denotes the $n$-th Laguerre polynomial.

- **Single-photon-added thermal state (SPATS):**
$$\chi_{\mathrm{SPATS}}(\beta) = \exp\!\left[-\frac{|\beta|^2}{2}\right]\left[1-(1+\bar n)|\beta|^2\right]\exp\!\left[-\bar n|\beta|^2\right]$$

- **Squeezed vacuum** with real squeezing parameter, $|r,0\rangle$:
$$\chi_{sv}(\beta) = \exp\!\left[-\frac{(\beta+\beta^*)^2 e^{2r}}{8} + \frac{(\beta-\beta^*)^2 e^{-2r}}{8}\right]$$

- **Superposition of coherent states**, $|\psi\rangle = \frac{1}{\sqrt N}\left(|\alpha\rangle + e^{i\theta}|-\alpha\rangle\right)$, with normalization $N = 2 + 2\cos(\theta)\,e^{-2|\alpha|^2}$:
$$\chi_\psi(\beta) = \frac{1}{N}\exp\!\left[-\frac{|\beta|^2}{2}\right]\Big[e^{\beta\alpha^*-\beta^*\alpha} + e^{i\theta}e^{\beta\alpha^*+\beta^*\alpha}e^{-2|\alpha|^2} + e^{-i\theta}e^{-\beta\alpha^*-\beta^*\alpha}e^{-2|\alpha|^2} + e^{-\beta\alpha^*+\beta^*\alpha}\Big]$$

## 1. Import libraries

In [ ]:
import numpy as np                                         # Numerical operations
from scipy.special import erf, eval_hermite, factorial     # Special functions
from scipy.optimize import bisect                          # Root finding (for inverse CDF)
import pandas as pd                                        # Data manipulation and storage
import matplotlib.pyplot as plt                            # Plotting
import time                                                # Timing code execution

## 2. Find and code the CDF

All CDF functions share the signature `cdf(x, phi, **parameters)`, so that the same inversion routine can be used for every state. For phase-independent states (Fock, SPATS) the argument `phi` is simply ignored. The functions also accept NumPy arrays for `x` and `phi`.

### Fock state

- Characteristic function:
$$\chi_n(\beta) = \exp\!\left[-\frac{|\beta|^2}{2}\right] L_n\!\left(|\beta|^2\right)$$

- Probability density:
$$p_n(x,\varphi) = \sum_{k=0}^{n}\frac{1}{\sqrt{2\pi}}\,\frac{n!}{2^k\,k!^2\,(n-k)!}\,\exp\!\left(-\frac{x^2}{2}\right) H_{2k}\!\left(\frac{x}{\sqrt 2}\right)$$

- Cumulative distribution function:
$$F_n(x,\varphi) = \frac12 + \frac12\,\mathrm{erf}\!\left(\frac{x}{\sqrt2}\right) - \frac{1}{\sqrt\pi}\exp\!\left(-\frac{x^2}{2}\right)\sum_{k=1}^{n}\frac{n!}{2^k\,k!^2\,(n-k)!}\,H_{2k-1}\!\left(\frac{x}{\sqrt2}\right)$$

In [ ]:
def fock_state_cdf(x, phi, fock_number):
    # phi is not used: Fock states are phase independent
    x = np.asarray(x, dtype=float)
    result = np.zeros_like(x)
    for k in range(1, fock_number + 1):
        coeff = factorial(fock_number) / (2**k * factorial(k)**2 * factorial(fock_number - k))
        result -= coeff * eval_hermite(2*k - 1, x / np.sqrt(2))
    result *= np.exp(-x**2 / 2) / np.sqrt(np.pi)
    result += 0.5 + 0.5 * erf(x / np.sqrt(2))
    return result

### SPATS

- Characteristic function:
$$\chi_{\mathrm{SPATS}}(\beta) = \exp\!\left[-\frac{|\beta|^2}{2}\right]\left[1-(1+\bar n)|\beta|^2\right]\exp\!\left[-\bar n|\beta|^2\right]$$

- Probability density:
$$p_{\mathrm{SPATS}}(x,\varphi) = \frac{1}{\sqrt{\pi(4\bar n+2)}}\exp\!\left(-\frac{x^2}{4\bar n+2}\right)\left[1-\frac{1+\bar n}{1+2\bar n}\left(1-\frac{x^2}{1+2\bar n}\right)\right]$$

- Cumulative distribution function:
$$F_{\mathrm{SPATS}}(x,\varphi) = \frac12 + \frac12\,\mathrm{erf}\!\left(\frac{x}{\sqrt{4\bar n+2}}\right) - \frac{x}{\sqrt{\pi(4\bar n+2)}}\,\frac{1+\bar n}{1+2\bar n}\,\exp\!\left(-\frac{x^2}{4\bar n+2}\right)$$

In [ ]:
def spats_cdf(x, phi, avg_pn):
    # phi is not used: SPATS is phase independent
    width = 4 * avg_pn + 2                       # = 2 * (variance of the underlying thermal state)
    scale = (1 + avg_pn) / (1 + 2 * avg_pn)
    return (0.5 + 0.5 * erf(x / np.sqrt(width))
            - x * scale / np.sqrt(np.pi * width) * np.exp(-x**2 / width))

### Squeezed vacuum with real squeezing parameter

- Characteristic function:
$$\chi_{sv}(\beta) = \exp\!\left[-\frac{(\beta+\beta^*)^2 e^{2r}}{8} + \frac{(\beta-\beta^*)^2 e^{-2r}}{8}\right]$$

- Probability density:
$$p_{sv}(x,\varphi) = \frac{1}{\sqrt{2\pi}}\,\frac{1}{\sqrt{e^{-2r}\cos^2\varphi + e^{2r}\sin^2\varphi}}\,\exp\!\left[-\frac{x^2}{2e^{-2r}\cos^2\varphi + 2e^{2r}\sin^2\varphi}\right]$$

- Cumulative distribution function:
$$F_{sv}(x,\varphi) = \frac12 + \frac12\,\mathrm{erf}\!\left[\frac{x}{\sqrt{2e^{-2r}\cos^2\varphi + 2e^{2r}\sin^2\varphi}}\right]$$

In [ ]:
def squeezed_state_cdf(x, phi, r):
    V_s = np.exp(-2 * r)   # Variance of the squeezed quadrature (phi = 0)
    V_a = np.exp(2 * r)    # Variance of the anti-squeezed quadrature (phi = pi/2)
    variance = V_s * np.cos(phi)**2 + V_a * np.sin(phi)**2
    return 0.5 + 0.5 * erf(x / np.sqrt(2 * variance))

### Superposition of coherent states

- Characteristic function:
$$\chi_\psi(\beta) = \frac{1}{N}\exp\!\left[-\frac{|\beta|^2}{2}\right]\Big[e^{\beta\alpha^*-\beta^*\alpha} + e^{i\theta}e^{\beta\alpha^*+\beta^*\alpha}e^{-2|\alpha|^2} + e^{-i\theta}e^{-\beta\alpha^*-\beta^*\alpha}e^{-2|\alpha|^2} + e^{-\beta\alpha^*+\beta^*\alpha}\Big]$$

- Probability density:
$$\begin{aligned}
p_\psi(x,\varphi) = \frac{1}{\sqrt{2\pi}}\,\frac{1}{N}\Bigg[ &\exp\!\left[-\frac{(\alpha e^{i\varphi}+\alpha^*e^{-i\varphi}-x)^2}{2}\right] + e^{i\theta}e^{-2|\alpha|^2}\exp\!\left[-\frac{(-\alpha e^{i\varphi}+\alpha^*e^{-i\varphi}-x)^2}{2}\right] \\
+\, &e^{-i\theta}e^{-2|\alpha|^2}\exp\!\left[-\frac{(\alpha e^{i\varphi}-\alpha^*e^{-i\varphi}-x)^2}{2}\right] + \exp\!\left[-\frac{(-\alpha e^{i\varphi}-\alpha^*e^{-i\varphi}-x)^2}{2}\right]\Bigg]
\end{aligned}$$

- Cumulative distribution function (each Gaussian term integrates to $\tfrac12[1+\mathrm{erf}(\cdot)]$, and the constants add up to $N$):
$$\begin{aligned}
F_\psi(x,\varphi) = \frac{1}{2N}\Bigg[ N &+ \mathrm{erf}\!\left(\frac{x-\alpha e^{i\varphi}-\alpha^*e^{-i\varphi}}{\sqrt2}\right) + e^{i\theta}e^{-2|\alpha|^2}\,\mathrm{erf}\!\left(\frac{x+\alpha e^{i\varphi}-\alpha^*e^{-i\varphi}}{\sqrt2}\right) \\
&+ e^{-i\theta}e^{-2|\alpha|^2}\,\mathrm{erf}\!\left(\frac{x-\alpha e^{i\varphi}+\alpha^*e^{-i\varphi}}{\sqrt2}\right) + \mathrm{erf}\!\left(\frac{x+\alpha e^{i\varphi}+\alpha^*e^{-i\varphi}}{\sqrt2}\right)\Bigg]
\end{aligned}$$

*Numerical note:* the complex-argument `erf` grows like $e^{2[\mathrm{Im}(\alpha e^{i\varphi})]^2}$ and overflows for $|\alpha| \gtrsim 19$. Moderate amplitudes ($|\alpha| \approx 2$) also show the interference features most clearly.

In [ ]:
def supcoh_state_cdf(x, phi, alpha, theta):
    N = 2 + 2 * np.cos(theta) * np.exp(-2 * np.abs(alpha)**2)
    c1 = np.exp(1j * theta) * np.exp(-2 * np.abs(alpha)**2)
    c2 = np.exp(-1j * theta) * np.exp(-2 * np.abs(alpha)**2)
    gamma = alpha * np.exp(1j * phi)

    result = (
        N
        + erf((x - 2 * np.real(gamma)) / np.sqrt(2))
        + c1 * erf((x + 2j * np.imag(gamma)) / np.sqrt(2))
        + c2 * erf((x - 2j * np.imag(gamma)) / np.sqrt(2))
        + erf((x + 2 * np.real(gamma)) / np.sqrt(2))
    )
    return np.real(result) / (2 * N)   # The imaginary part is always zero

## 3. Invert the CDF to find the quadrature value

We generate values of the quadrature $\hat X_\varphi$ with the desired distribution using the **inversion method** (inverse transform sampling): for any real random variable $X$, the random variable $F_X^{-1}(U)$ has the same distribution as $X$, where $F_X^{-1}$ is the generalized inverse of the CDF $F_X$ of $X$ and $U$ is uniform on $[0,1]$.

We also need to set up some parameters with the desired values.

*Runtime note:* the loop below calls `bisect` once per sample in pure Python. With $10^5$ samples this takes about 20 s for the squeezed vacuum and several minutes for Fock states; $10^6$ samples (as used for the figures in the lecture notes) take ten times longer. A vectorized alternative is given in the next cell.

In [ ]:
np.random.seed(42)                                         # For reproducibility
num_samples = 1_000_000                                      # Number of samples to generate
tolerance = 1e-12                                          # Tolerance for numerical methods
phases = 2 * np.pi * np.random.rand(num_samples) - np.pi   # Random phases in [-pi, pi)
target_cdf_values = np.random.rand(num_samples)            # Random numbers in [0, 1)
search_width = 50   # Root search interval [-search_width, search_width]; must exceed 2|alpha| + a few std devs
sample_values = np.zeros(num_samples)                      # Array to hold sampled quadrature values


def cdf_inversion(sample_values, target_cdf_values, phases, cdf_func, arguments, tolerance):
    num_samples = len(sample_values)
    start_time = time.time()
    last_report = start_time

    for i in range(num_samples):
        sample_values[i] = bisect(
            lambda x: cdf_func(x, phases[i], **arguments) - target_cdf_values[i],
            -search_width, search_width, xtol=tolerance
        )

        now = time.time()
        if now - last_report > 10:
            print(f'Progress: {i+1}/{num_samples} ({100*(i+1)/num_samples:.1f}%) '
                  f'time elapsed: {now - start_time:.2f} seconds.')
            last_report = now

    print(f'Progress: {i+1}/{num_samples} ({100*(i+1)/num_samples:.1f}%)')
    print(f'CDF inversion completed in {time.time() - start_time:.2f} seconds.')
    return sample_values

### Optional: vectorized inversion

The same bisection, applied to all samples at once with NumPy arrays. It gives the same result in a few seconds, even for $10^6$ samples.

In [ ]:
def cdf_inversion_vectorized(target_cdf_values, phases, cdf_func, arguments, tolerance):
    start_time = time.time()
    low = np.full(len(target_cdf_values), -float(search_width))
    high = np.full(len(target_cdf_values), float(search_width))
    while np.max(high - low) > tolerance:
        mid = 0.5 * (low + high)
        below = cdf_func(mid, phases, **arguments) < target_cdf_values
        low = np.where(below, mid, low)
        high = np.where(below, high, mid)
    print(f'Vectorized CDF inversion completed in {time.time() - start_time:.2f} seconds.')
    return 0.5 * (low + high)

## 4. Use the function!

Now we can execute the function with the corresponding parameters to simulate the desired state. Choose the state by setting `cdf_func` and `arguments`.

In [ ]:
fock_arguments = {
    'fock_number': 3
}

spats_arguments = {
    'avg_pn': 3
}

sv_arguments = {
    'r': 1
}

supcoh_arguments = {
    'alpha': 2 + 1j,
    'theta': 0
}

cdf_func = supcoh_state_cdf
arguments = supcoh_arguments

use_vectorized = True   # Set to True for the fast version

# Sanity check: the CDF must go from 0 to 1 inside the search interval
print('F(-search_width) =', np.max(cdf_func(-search_width, phases[:1000], **arguments)))
print('F(+search_width) =', np.min(cdf_func(search_width, phases[:1000], **arguments)))

if use_vectorized:
    sample_values = cdf_inversion_vectorized(target_cdf_values, phases, cdf_func, arguments, tolerance)
else:
    sample_values = cdf_inversion(sample_values, target_cdf_values,
                                  phases, cdf_func, arguments, tolerance)

df = pd.DataFrame({'sample_value': sample_values, 'phase': phases})
df.head()

In [ ]:
# Plotting the cdf
x_values = np.linspace(-10, 10, 1000)
cdf_values = cdf_func(x_values, 0, alpha=2 + 1j, theta=1)  # Using phase=0 for plotting
plt.figure(figsize=(10, 6))
plt.plot(x_values, cdf_values, label='Superposition of Coherent States (α=2+0.2i, θ=π/4)')
plt.title('Cumulative Distribution Functions of Various Quantum States')
plt.xlabel('Quadrature Value')
plt.ylabel('Cumulative Probability')
plt.legend()
plt.grid()
plt.show()

## 5. Evaluation and data analysis

### Task 2
Check whether the data sets represent physically correct states according to Heisenberg's uncertainty relation. Considering the commutator $[\hat X_\varphi, \hat X_{\varphi+\pi/2}] = -2i$, it follows that

$$\mathrm{Var}[\hat X_\varphi]\cdot\mathrm{Var}[\hat X_{\varphi+\pi/2}] \ge 1.$$

Discuss the variances of the quadrature operator. The analytical expressions are:

| State | $\langle(\Delta\hat X_\varphi)^2\rangle$ |
|---|---|
| Fock | $2n+1$ |
| SPATS | $4\bar n+3$ |
| Squeezed vacuum | $\lvert e^{i\varphi}\cosh r - e^{-i\varphi}\sinh r\rvert^2$ |
| Superposition of coherent states | $(\alpha e^{i\varphi}-\alpha^*e^{-i\varphi})^2\left[1+\frac{4\sin^2\theta}{N^2}e^{-4\lvert\alpha\rvert^2}\right] + \frac{4\lvert\alpha\rvert^2}{1+e^{-2\lvert\alpha\rvert^2}\cos\theta} + 1$ |

Note that $(\alpha e^{i\varphi}-\alpha^*e^{-i\varphi})^2 = -4\left[\mathrm{Im}(\alpha e^{i\varphi})\right]^2 \le 0$.

**Statistical uncertainty.** A variance estimated from $M$ samples has a standard error of about $\mathrm{Var}\sqrt{2/(M-1)}$. The squeezed vacuum is a minimum-uncertainty state: its product equals exactly 1 at $\varphi = 0, \pm\pi/2$. There, the estimated product will fall *below* 1 about half of the time purely through sampling noise. A violation is only meaningful if it is larger than a few standard errors.

In [ ]:
# Theoretical variance
def fock_state_var_th(fock_number):
    return 2 * fock_number + 1

def spats_var_th(avg_pn):
    return 4 * avg_pn + 3

def squeezed_state_var_th(phi, r):
    return np.abs(np.exp(1j*phi)*np.cosh(r) - np.exp(-1j*phi)*np.sinh(r))**2

def supcoh_state_var_th(phi, alpha, theta):
    N = 2 + 2 * np.exp(-2 * np.abs(alpha)**2) * np.cos(theta)
    k0 = alpha * np.exp(1j * phi)
    k1 = 1 + (2 * np.exp(-2 * np.abs(alpha)**2) * np.sin(theta) / N)**2
    k2 = (4 * np.abs(alpha)**2) / (N / 2)
    return np.real((2j * np.imag(k0))**2 * k1 + k2 + 1)


def compute_variance_by_state(cdf_func, sample_values, phases, arguments, num_bins=100):
    '''
    Compute the variance for fock/spats (single value) or sv/supcoh (per phase bin).
    Returns a DataFrame with columns: phase, counts, theoretical_variance,
    calculated_variance, std_error
    '''
    if cdf_func in [fock_state_cdf, spats_cdf]:
        # Single variance, phase not relevant
        if cdf_func == fock_state_cdf:
            theoretical = fock_state_var_th(arguments['fock_number'])
        else:
            theoretical = spats_var_th(arguments['avg_pn'])
        M = len(sample_values)
        calculated = np.var(sample_values)
        return pd.DataFrame({'phase': [None], 'counts': [M],
                             'theoretical_variance': [theoretical],
                             'calculated_variance': [calculated],
                             'std_error': [calculated * np.sqrt(2 / (M - 1))]})
    else:
        # Per phase bin
        bins = np.linspace(-np.pi, np.pi, num_bins + 1)
        bin_centers = 0.5 * (bins[:-1] + bins[1:])
        counts = np.zeros(num_bins, dtype=int)
        theoretical = np.full(num_bins, np.nan)
        calculated = np.full(num_bins, np.nan)
        for i in range(num_bins):
            mask = (phases >= bins[i]) & (phases < bins[i+1])
            counts[i] = np.sum(mask)
            if counts[i] > 1:
                calculated[i] = np.var(sample_values[mask])
                if cdf_func == squeezed_state_cdf:
                    theoretical[i] = squeezed_state_var_th(bin_centers[i], arguments['r'])
                else:
                    theoretical[i] = supcoh_state_var_th(bin_centers[i], arguments['alpha'],
                                                         arguments['theta'])
        std_error = calculated * np.sqrt(2 / np.maximum(counts - 1, 1))
        return pd.DataFrame({'phase': bin_centers, 'counts': counts,
                             'theoretical_variance': theoretical,
                             'calculated_variance': calculated,
                             'std_error': std_error})


variance_df = compute_variance_by_state(cdf_func, sample_values, phases, arguments, num_bins=100)
print('Computed variance:\n', variance_df)

In [ ]:
# Check the Heisenberg uncertainty relation
if len(variance_df) > 1:
    # Phase-dependent states: multiply the variances at phi and phi + pi/2.
    # The variance is pi-periodic and the bins cover [-pi, pi), so wrapping around is allowed.
    bin_width = np.mean(np.diff(variance_df['phase']))
    shift = int(round((np.pi / 2) / bin_width))

    var = variance_df['calculated_variance'].to_numpy()
    err = variance_df['std_error'].to_numpy()
    shifted_var = np.roll(var, -shift)
    shifted_err = np.roll(err, -shift)

    variance_df['shifted_variance'] = shifted_var
    variance_df['variance_product'] = var * shifted_var
    variance_df['product_error'] = (variance_df['variance_product']
                                    * np.sqrt((err / var)**2 + (shifted_err / shifted_var)**2))
    print('Variance product for Heisenberg relation:\n',
          variance_df[['phase', 'calculated_variance', 'shifted_variance',
                       'variance_product', 'product_error']])
else:
    # Phase-independent states: Var[X_phi] * Var[X_phi + pi/2] = Var^2
    variance_df['variance_product'] = variance_df['calculated_variance']**2
    variance_df['product_error'] = 2 * variance_df['variance_product'] * (
        variance_df['std_error'] / variance_df['calculated_variance'])
    print('Variance product for Heisenberg relation:\n',
          variance_df[['calculated_variance', 'variance_product', 'product_error']])

# Is the bound violated by more than 3 standard errors anywhere?
significance = (variance_df['variance_product'] - 1) / variance_df['product_error']
print(f"\nSmallest product: {variance_df['variance_product'].min():.4f}")
print(f"Minimum of (product - 1) / error: {significance.min():.2f}  (below -3 would signal a violation)")
print('Uncertainty relation violated (> 3 sigma):', bool((significance < -3).any()))

### Task 3
Visualize the data sets in $x$–$\varphi$ diagrams (linear and/or polar). Discuss the correlations between the randomly generated quadrature values $x$ and phases $\varphi$ in the cases of physical and non-physical states, and compare with the vacuum state. Check the phase and the quadrature distributions through their histograms.

*About the polar plot:* Matplotlib's polar axes do not mirror negative radii; they shift the radial origin to the smallest value instead, which creates an artificial hole in the center. We therefore plot the points $(x\cos\varphi,\ x\sin\varphi)$ on ordinary axes with equal aspect ratio, which gives the intended polar picture.

In [ ]:
# Vacuum data for comparison (squeezed state with r = 0), using fresh random numbers
vacuum_targets = np.random.rand(num_samples)
vacuum_phases = 2 * np.pi * np.random.rand(num_samples) - np.pi
vacuum_sample_values = cdf_inversion_vectorized(vacuum_targets, vacuum_phases,
                                                squeezed_state_cdf, {'r': 0}, tolerance)

In [ ]:
# Linear scatter plot of quadrature vs phase
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
axes[0].scatter(phases, sample_values, s=1, alpha=0.5)
axes[0].set_title('Simulated state')
axes[1].scatter(vacuum_phases, vacuum_sample_values, s=1, alpha=0.5, color='tab:gray')
axes[1].set_title('Vacuum')
for ax in axes:
    ax.set_xlabel('Phase $\\phi$')
    ax.grid(True)
axes[0].set_ylabel('Quadrature $x$')
fig.suptitle('Scatter plot: Quadrature vs Phase')
plt.show()

# Polar representation: point (x cos(phi), x sin(phi))
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
L = 1.05 * np.max(np.abs(sample_values))   # common square plot range
axes[0].scatter(sample_values * np.cos(phases), sample_values * np.sin(phases), s=1, alpha=0.3)
axes[0].set_title('Simulated state')
axes[1].scatter(vacuum_sample_values * np.cos(vacuum_phases),
                vacuum_sample_values * np.sin(vacuum_phases), s=1, alpha=0.3, color='tab:gray')
axes[1].set_title('Vacuum')
for ax in axes:
    ax.set_xlim(-L, L)
    ax.set_ylim(-L, L)
    ax.set_aspect('equal')
    ax.set_xlabel('$x\\cos\\phi$')
    ax.grid(True)
axes[0].set_ylabel('$x\\sin\\phi$')
fig.suptitle('Polar plot: Quadrature vs Phase')
plt.show()

# Histogram of quadrature values
plt.figure(figsize=(8, 5))
plt.hist(sample_values, bins=100, alpha=0.7, color='tab:blue', density=True, label='Simulated state')
plt.hist(vacuum_sample_values, bins=100, alpha=0.5, color='tab:gray', density=True, label='Vacuum')
plt.xlabel('Quadrature $x$')
plt.ylabel('Probability density')
plt.title('Histogram of Quadrature Values')
plt.legend()
plt.grid(True)
plt.show()

# Histogram of phase values
plt.figure(figsize=(8, 5))
plt.hist(phases, bins=100, alpha=0.7, color='tab:orange')
plt.xlabel('Phase $\\phi$')
plt.ylabel('Counts')
plt.title('Histogram of Phase Values')
plt.grid(True)
plt.show()

# Plot of variance vs phase
if len(variance_df) > 1:
    plt.figure(figsize=(8, 5))
    plt.errorbar(variance_df['phase'], variance_df['calculated_variance'],
                 yerr=variance_df['std_error'], fmt='o', markersize=3,
                 label='Calculated Variance', color='tab:blue')
    plt.plot(variance_df['phase'], variance_df['theoretical_variance'],
             label='Theoretical Variance', color='tab:orange')
    plt.xlabel('Phase $\\phi$')
    plt.ylabel('Variance')
    plt.title('Variance vs Phase')
    plt.legend()
    plt.grid(True)
    plt.show()

### Optional: a non-physical example

To see what Task 2 detects, simulate a Gaussian "state" whose quadratures are *both* squeezed, $\mathrm{Var}[\hat X_\varphi] = e^{-2r}\cos^2\varphi + e^{-2s}\sin^2\varphi$ with $r, s > 0$. No quantum state has these statistics, because the product of the variances at $\varphi = 0$ and $\varphi = \pi/2$ is $e^{-2(r+s)} < 1$.

In [ ]:
def nonphysical_cdf(x, phi, r, s):
    variance = np.exp(-2 * r) * np.cos(phi)**2 + np.exp(-2 * s) * np.sin(phi)**2
    return 0.5 + 0.5 * erf(x / np.sqrt(2 * variance))

np_args = {'r': 0.5, 's': 0.5}
np_samples = cdf_inversion_vectorized(target_cdf_values, phases, nonphysical_cdf, np_args, tolerance)

bins = np.linspace(-np.pi, np.pi, 101)
centers = 0.5 * (bins[:-1] + bins[1:])
np_var = np.array([np.var(np_samples[(phases >= bins[i]) & (phases < bins[i+1])])
                   for i in range(100)])
np_product = np_var * np.roll(np_var, -25)   # 25 bins = pi/2
print(f'Largest variance product: {np_product.max():.3f}  (must be >= 1 for a physical state)')

plt.figure(figsize=(8, 5))
plt.plot(centers, np_product, 'o', markersize=3)
plt.axhline(1, color='red', linestyle='--', label='Heisenberg bound')
plt.xlabel('Phase $\\phi$')
plt.ylabel('Var($\\phi$) Var($\\phi+\\pi/2$)')
plt.title('Non-physical example')
plt.legend()
plt.grid(True)
plt.show()

### Task 4
If the previous analysis indicates squeezing, prove it with the squeezing condition: the normally ordered variance must be negative,

$$\langle :(\Delta\hat X_\varphi)^2: \rangle = \langle(\Delta\hat X_\varphi)^2\rangle - \langle 0|(\Delta\hat X_\varphi)^2|0\rangle < 0.$$

The vacuum variance is exactly 1 in our convention. The vacuum samples from Task 3 provide a consistency check of this value.

In [ ]:
# Vacuum variance: exact value (squeezed state with r = 0) and check with the simulated vacuum data
vacuum_variance = squeezed_state_var_th(0, 0)    # This is exactly 1
print(f'Vacuum state variance: {vacuum_variance}')
print(f'Calculated vacuum state variance from samples: {np.var(vacuum_sample_values):.4f}')

if len(variance_df) > 1:
    variance_df['vacuum_variance'] = vacuum_variance
    variance_df['normal_ordered_variance'] = (variance_df['calculated_variance']
                                              - vacuum_variance)
    print('Variance with normal ordering:\n',
          variance_df[['phase', 'calculated_variance', 'vacuum_variance',
                       'normal_ordered_variance', 'std_error']])

    # Smallest normally ordered variance and its significance
    i_min = variance_df['normal_ordered_variance'].idxmin()
    row = variance_df.loc[i_min]
    print(f"\nMinimum normally ordered variance: {row['normal_ordered_variance']:.4f} "
          f"+/- {row['std_error']:.4f} at phi = {row['phase']:.3f}, "
          f"i.e. {row['normal_ordered_variance'] / row['std_error']:.1f} standard errors from 0 "
          f"(below -3 is a significant sign of squeezing)")

    # Plot the normally ordered variance
    plt.figure(figsize=(8, 5))
    plt.errorbar(variance_df['phase'], variance_df['normal_ordered_variance'],
                 yerr=variance_df['std_error'], fmt='o', markersize=3,
                 label='Normally Ordered Variance', color='tab:green')
    plt.axhline(0, color='red', linestyle='--', label='Zero Line')
    plt.xlabel('Phase $\\phi$')
    plt.ylabel('Normal Ordered Variance')
    plt.title('Normally Ordered Variance vs Phase')
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print('Phase-independent state: normally ordered variance =',
          float(variance_df['calculated_variance'][0] - vacuum_variance))